###### Imports and Settings

In [3]:
import pandas as pd
#import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [4]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains ACS 5-Year Estimate variables. This document does contain one variable from the ACS 5-Year Subject Tables: S0801_C01_046E, mean travel-time to work.

In [5]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [6]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']
#api_key = '24fc7d81b74510d599f702dbd408fb18e1466d81'

In [7]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828 
+ dg19: ID's 829 - 874  
+ dg20: ID's 875 - 920  
+ dg21: ID's 921 - 966  
+ dg22: ID's 967 - 1012  
+ dg23: ID's 1013 - 1058  
+ dg24: ID's 1059 - 1104  
+ dg25: ID's 1105 - 1150  
+ dg26: ID's 1151 - 1196  
+ dg27: ID's 1197 - 1242

**This data guide stops at ID 1203 as of 6/28/2022.**

In [140]:
dataguide = pd.read_csv('../data guides/DATA GUIDE ACS.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [141]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]
dg14 = dataguide[dataguide['ID'].between(599, 644)]
dg15 = dataguide[dataguide['ID'].between(645, 690)]
dg16 = dataguide[dataguide['ID'].between(691, 736)]
dg17 = dataguide[dataguide['ID'].between(737, 782)]
dg18 = dataguide[dataguide['ID'].between(783, 828)]
dg19 = dataguide[dataguide['ID'].between(829, 874)]
dg20 = dataguide[dataguide['ID'].between(875, 920)]
dg21 = dataguide[dataguide['ID'].between(921, 966)]
dg22 = dataguide[dataguide['ID'].between(967, 1012)]
dg23 = dataguide[dataguide['ID'].between(1013, 1058)]
dg24 = dataguide[dataguide['ID'].between(1059, 1104)]
dg25 = dataguide[dataguide['ID'].between(1105, 1150)]
dg26 = dataguide[dataguide['ID'].between(1151, 1196)]
dg27 = dataguide[dataguide['ID'].between(1197, 1242)]

## One Through Twenty Seven

In [215]:
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)

url_str= 'https://api.census.gov/data/2009/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)

In [212]:
places.head(503)

,NAME,GEO_ID,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,StateFIPS,GeoFIPS
0,"Mount Carmel town, Tennessee",1600000US4750580,5356,2537,121,102,185,49,123,49,76,53,140,137,123,138,255,183,245,71,106,70,77,122,88,24,0,2819,212,84,85,147,0,54,63,53,109,139,226,238,346,282,118,69,117,88,66,208,47,50580
1,"Mount Juliet city, Tennessee",1600000US4750780,20501,10255,831,991,773,476,233,204,64,307,920,884,856,809,898,606,520,178,149,144,109,157,36,56,54,10246,1072,685,703,612,255,83,134,305,699,1015,845,639,722,761,538,173,122,99,213,148,47,50780
2,"Mount Pleasant city, Tennessee",1600000US4751080,4451,2194,199,168,150,162,160,12,0,120,41,309,50,122,38,165,119,27,84,19,47,88,70,31,13,2257,43,129,144,129,111,21,66,47,95,174,77,207,193,202,119,31,40,58,31,163,47,51080
3,"Munford city, Tennessee",1600000US4751540,6304,3006,173,143,334,145,61,25,0,196,187,93,182,313,207,205,304,13,87,16,79,87,83,59,14,3298,144,222,336,102,63,0,20,198,132,191,278,230,390,271,213,15,116,31,58,124,47,51540
4,"Murfreesboro city, Tennessee",1600000US4751560,97423,48072,3274,3186,3159,1898,2247,1607,1060,3016,6103,4099,3157,3384,3010,2365,1797,632,694,506,672,809,542,501,354,49351,3002,3132,3101,1672,2323,1531,1211,2807,5846,3875,3315,3470,2973,2514,2051,655,953,580,912,959,47,51560
5,Nashville-Davidson metropolitan government (ba...,1600000US4752006,592497,287475,22375,17907,16503,9783,8564,4744,4641,13540,29985,25245,22516,20956,20648,18703,15587,5446,5966,3451,4167,6065,5002,3242,2439,305022,21743,17637,16036,9641,8742,3955,4714,14374,30096,25711,20807,21214,21277,20031,18004,5550,7112,4033,5236,8654,47,52006
6,"Newbern town, Tennessee",1600000US4752400,3117,1316,105,67,194,61,11,0,18,22,121,127,44,148,49,85,35,66,29,18,30,18,44,0,24,1801,166,72,162,81,121,0,11,23,124,142,136,99,160,54,97,25,48,19,0,86,47,52400
7,"New Hope city, Tennessee",1600000US4752780,923,426,36,23,20,18,15,2,10,19,10,23,30,47,51,20,47,17,3,0,11,5,8,6,5,497,34,39,48,43,9,1,0,7,32,29,33,39,39,46,40,8,12,0,4,16,47,52780
8,"New Johnsonville city, Tennessee",1600000US4752820,1694,869,43,52,51,47,20,2,13,4,37,42,94,94,45,43,111,12,56,14,30,33,8,10,8,825,32,50,44,27,18,2,4,22,34,42,102,55,34,82,93,25,37,11,31,35,47,52820
9,"New Market town, Tennessee",1600000US4752940,1431,826,60,66,31,17,31,7,0,26,64,55,84,99,50,81,41,17,21,14,2,22,25,13,0,605,4,24,81,4,4,2,5,12,42,41,52,43,73,33,62,7,42,17,4,13,47,52940


## Places Needed:  
### Per County:  
+ Cheatham County: Ashland City town, Kingston Springs town, Pegram town, Pleasant View city  
+ Davidson County: Belle Meade city, Berry Hill city, Forest Hills city, **Goodlettsville** city, 
Nashville-Davidson metropolitan government (balance), Oak Hill city, **Ridgetop** city  
+ Dickson County: Burns town, Charlotte town, Dickson city, Slayden town, Vanleer town, White Bluff town  
+ Houston County: Erin city, **Tennessee Ridge** town  
+ Humphreys County: McEwen city, New Johnsonville city, Waverly city  
+ Maury County: Columbia city, Mount Pleasant city, **Spring Hill** city  
+ Montgomery County: Clarksville city  
+ Robertson County: Adams city, Cedar Hill city, Coopertown town, Cross Plains city, Greenbrier town, **Millersville** city, **Portland** city, **Ridgetop** city, Springfield city, **White House** city  
+ Rutherford County: Eagleville city, La Vergne city, Murfreesboro city, Smyrna town
+ Stewart County: Cumberland City town, Dover city, **Tennessee Ridge** town  
+ Sumner County: Gallatin city, **Goodlettsville** city, Hendersonville city, **Millersville** city, **Portland** city, Westmoreland town, **White House** city  
+ Trousdale County: Hartsville  
+ Williamson County: Brentwood city, Fairview city, Franklin city, Nolensville town, **Spring Hill** city, Thompson's Station town  
+ Wilson County: Lebanon city, Mount Juliet city, Watertown city  

So the ones without overlapping places are: Cheatham, Dickson, Humphreys, Montgomery, Rutherford, Trousdale, and Wilson Counties  

In [217]:
lakewood = places.loc[places['NAME'] == "Nashville-Davidson metropolitan government (balance), Tennessee"]
lakewood

,NAME,GEO_ID,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,StateFIPS,GeoFIPS
5,Nashville-Davidson metropolitan government (ba...,1600000US4752006,592497,287475,22375,17907,16503,9783,8564,4744,4641,13540,29985,25245,22516,20956,20648,18703,15587,5446,5966,3451,4167,6065,5002,3242,2439,305022,21743,17637,16036,9641,8742,3955,4714,14374,30096,25711,20807,21214,21277,20031,18004,5550,7112,4033,5236,8654,47,52006


In [214]:
places = ['1600000US4702180', #Ashland City: Cheatham
          '1600000US4739660', #Kingston Springs: Cheatham
          '1600000US4757480', #Pegram: Cheatham
          '1600000US4759560', #Pleasant View: Cheatham
          '1600000US4704620', #Belle Meade: Davidson
          '1600000US4705140', #Berry Hill: Davidson
          '1600000US4727020', #Forest Hills: Davidson
          '1600000US4729920', #Goodlettsville: Davidson/Sumner
          '1600000US4752006', #Nashville-Davidson metropolitan government (balance): Davidson
          '1600000US4754780', #Oak Hill: Davidson
          '1600000US4763140', #Ridgetop: Davidson/Robertson
          '1600000US4709880', #Burns: Dickson
          '1600000US4713080', #Charlotte: Dickson
          '1600000US4720620', #Dickson: Dickson
          '1600000US4769080', #Slayden: Dickson
          '1600000US4776860', #Vanleer: Dickson
          '1600000US4779980', #White Bluff: Dickson
          '1600000US4724320', #Erin: Houston
          '1600000US4773460', #Tennessee Ridge: Houston/Stewart
          '1600000US4744840', #McEwen: Humphreys
          '1600000US4752820', #New Johnsonville: Humphreys
          '1600000US4778560', #Waverly: Humphreys
          '1600000US4716540', #Columbia: Maury
          '1600000US4751080', #Mount Pleasant: Maury
          '1600000US4770580', #Spring Hill: Maury/Williamson
          '1600000US4715160', #Clarksville: Montgomery
          '1600000US4700200', #Adams: Robertson
          '1600000US4711980', #Cedar Hill: Robertson
          '1600000US4716980', #Coopertown: Robertson
          '1600000US4718420', #Cross Plains: Robertson
          '1600000US4730960', #Greenbrier: Robertson
          '1600000US4748980', #Millersville: Robertson/Sumner
          '1600000US4760280', #Portland: Robertson/Sumner
          '1600000US4770500', #Springfield: Robertson
          '1600000US4780200', #White House: Robertson/Sumner
          '1600000US4722360', #Eagleville: Rutherford
          '1600000US4741200', #La Vergne: Rutherford
          '1600000US4751560', #Murfreesboro: Rutherford
          '1600000US4769420', #Smyrna: Rutherford
          '1600000US4718820', #Cumberland City: Stewart
          '1600000US4721400', #Dover: Stewart
          '1600000US4728540', #Gallatin: Sumner
          '1600000US4733280', #Hendersonville: Sumner
          '1600000US4779420', #Westmoreland: Sumner
          '1600000US4708280', #Brentwood: Williamson
          '1600000US4725440', #Fairview: Williamson
          '1600000US4727740', #Franklin: Williamson
          '1600000US4753460', #Nolensville: Williamson
          '1600000US4773900', #Thompson's Station: Williamson
          '1600000US4741520', #Lebanon: Wilson
          '1600000US4750780', #Mount Juliet: Wilson
          '1600000US4778320'] #Watertown: Wilson


In [37]:
test=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')

In [39]:
# ONE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
onepull = one
print('Okay Finished')

Okay Finished


In [40]:
#placemarker
one = onepull

In [41]:
# TWO
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
twopull = two
print('Okay Finished')

Okay Finished


In [42]:
two = twopull

In [43]:
# THREE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg3['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg3['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
three = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
three = three.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
three = three.append(state, ignore_index = True)
threepull = three
print('Okay Finished')

Okay Finished


In [44]:
three = threepull

In [45]:
# FOUR
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg4['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg4['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
four = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
four = four.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
four = four.append(state, ignore_index = True)
fourpull = four
print('Okay Finished')

Okay Finished


In [46]:
four = fourpull

In [47]:
# FIVE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg5['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg5['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
five = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
five = five.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
five = five.append(state, ignore_index = True)
fivepull = five
print('Okay Finished')

Okay Finished


In [48]:
five = fivepull

In [49]:
# SIX
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg6['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg6['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
six = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
six = six.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
six = six.append(state, ignore_index = True)
sixpull = six
print('Okay Finished')

Okay Finished


In [50]:
six = sixpull

In [51]:
# SEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg7['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg7['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
seven = seven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seven = seven.append(state, ignore_index = True)
sevenpull = seven
print('Okay Finished')

Okay Finished


In [52]:
seven = sevenpull

In [53]:
# EIGHT
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg8['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg8['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eight = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
eight = eight.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eight = eight.append(state, ignore_index = True)
eightpull = eight
print('Okay Finished')

Okay Finished


In [54]:
eight = eightpull

In [55]:
# NINE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg9['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg9['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
nine = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
nine = nine.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
nine = nine.append(state, ignore_index = True)
ninepull = nine
print('Okay Finished')

Okay Finished


In [56]:
nine = ninepull

In [57]:
# TEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg10['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg10['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
ten = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
ten = ten.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
ten = ten.append(state, ignore_index = True)
tenpull = ten
print('Okay Finished')

Okay Finished


In [58]:
ten = tenpull

In [59]:
# ELEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg11['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg11['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eleven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
eleven = eleven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eleven = eleven.append(state, ignore_index = True)
elevenpull = eleven
print('Okay Finished')

Okay Finished


In [61]:
eleven = elevenpull

In [62]:
# Twelve
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg12['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg12['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twelve = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twelve = twelve.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twelve = twelve.append(state, ignore_index = True)
twelvepull = twelve
print('Okay Finished')

Okay Finished


In [63]:
twelve = twelvepull

In [64]:
# Thirteen
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg13['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg13['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
thirteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
thirteen = thirteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
thirteen = thirteen.append(state, ignore_index = True)
thirteenpull = thirteen
print('Okay Finished')

Okay Finished


In [65]:
thirteen = thirteenpull

In [66]:
# FOURTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg14['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg14['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
fourteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
fourteen = fourteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
fourteen = fourteen.append(state, ignore_index = True)
fourteenpull = fourteen
print('Okay Finished')

Okay Finished


In [67]:
fourteen = fourteenpull

In [68]:
# FIFTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg15['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg15['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
fifteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
fifteen = fifteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
fifteen = fifteen.append(state, ignore_index = True)
fifteenpull = fifteen
print('Okay Finished')

Okay Finished


In [69]:
fifteen = fifteenpull

In [70]:
# SIXTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg16['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg16['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
sixteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
sixteen = sixteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
sixteen = sixteen.append(state, ignore_index= True)
sixteenpull = sixteen
print('Okay Finished')

Okay Finished


In [71]:
sixteen = sixteenpull

In [72]:
# SEVENTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg17['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg17['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seventeen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
seventeen = seventeen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seventeen = seventeen.append(state, ignore_index= True)
seventeenpull = seventeen
print('Okay Finished')

Okay Finished


In [73]:
seventeen = seventeenpull

In [74]:
# EIGHTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg18['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg18['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eighteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
eighteen = eighteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eighteen = eighteen.append(state, ignore_index= True)
eighteenpull = eighteen
print('Okay Finished')

Okay Finished


In [75]:
eighteen = eighteenpull

In [76]:
# NINETEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg19['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg19['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
nineteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
nineteen = nineteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
nineteen = nineteen.append(state, ignore_index= True)
nineteenpull = nineteen
print('Okay Finished')

Okay Finished


In [77]:
nineteen = nineteenpull

In [78]:
# TWENTY
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg20['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg20['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twenty = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twenty = twenty.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twenty = twenty.append(state, ignore_index= True)
twentypull = twenty
print('Okay Finished')

Okay Finished


In [79]:
twenty = twentypull

In [80]:
# TWENTYONE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg21['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg21['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentyone = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentyone = twentyone.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentyone = twentyone.append(state, ignore_index= True)
twentyonepull = twentyone
print('Okay Finished')

Okay Finished


In [81]:
twentyone = twentyonepull

In [82]:
# TWENTYTWO
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg22['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg22['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentytwo = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentytwo = twentytwo.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentytwo = twentytwo.append(state, ignore_index= True)
twentytwopull = twentytwo
print('Okay Finished')

Okay Finished


In [83]:
twentytwo = twentytwopull

In [84]:
# TWENTYTHREE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg23['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg23['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentythree = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentythree = twentythree.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentythree = twentythree.append(state, ignore_index= True)
twentythreepull = twentythree
print('Okay Finished')

Okay Finished


In [85]:
twentythree = twentythreepull

In [86]:
# TWENTYFOUR
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg24['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg24['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentyfour = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentyfour = twentyfour.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentyfour = twentyfour.append(state, ignore_index= True)
twentyfourpull = twentyfour
print('Okay Finished')

Okay Finished


In [87]:
twentyfour = twentyfourpull

In [88]:
# TWENTYFIVE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg25['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg25['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentyfive = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentyfive = twentyfive.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentyfive = twentyfive.append(state, ignore_index= True)
twentyfivepull = twentyfive
print('Okay Finished')

Okay Finished


In [89]:
twentyfive = twentyfivepull

In [90]:
# TWENTYSIX
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg26['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg26['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentysix = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentysix = twentysix.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentysix = twentysix.append(state, ignore_index= True)
twentysixpull = twentysix
print('Okay Finished')

Okay Finished


In [91]:
twentysix = twentysixpull

In [92]:
# TWENTYSEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg27['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg27['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twentyseven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                |(places['GEO_ID']=='1600000US4741520')|(places['GEO_ID']=='1600000US4702180')|(places['GEO_ID']=='1600000US4702180')
                |(places['GEO_ID']=='1600000US4739660')|(places['GEO_ID']=='1600000US4757480')|(places['GEO_ID']=='1600000US4759560')
                |(places['GEO_ID']=='1600000US4709880')|(places['GEO_ID']=='1600000US4713080')|(places['GEO_ID']=='1600000US4720620')
                |(places['GEO_ID']=='1600000US4769080')|(places['GEO_ID']=='1600000US4776860')|(places['GEO_ID']=='1600000US4779980')
                |(places['GEO_ID']=='1600000US4744840')|(places['GEO_ID']=='1600000US4752820')|(places['GEO_ID']=='1600000US4778560')
                |(places['GEO_ID']=='1600000US4715160')|(places['GEO_ID']=='1600000US4732742')|(places['GEO_ID']=='1600000US4728540')]
twentyseven = twentyseven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twentyseven = twentyseven.append(state, ignore_index= True)
twentysevenpull = twentyseven
print('Okay Finished')

Okay Finished


In [93]:
twentyseven = twentysevenpull

In [94]:
last = twentyseven

In [139]:
last.tail()

,GEO_ID,traveltimemode_taximotorcyclebicycleorother_45to59,traveltimemode_taximotorcyclebicycleorother_60ormore,fb_total_series,fb_nativeborn,fb_foreignborn,fb_foreignborn_naturalizeduscitizen,fb_foreignborn_notauscitizen,disability_agebynumber_series_total,disability_u18_total,disability_u18_1disability,disability_u18_2ormoredisabilities,disability_u18_nodisability,disability_18to64_total,disability_18to64_1disability,disability_18to64_2ormoredisabilities,disability_18to64_nodisability,disability_65+_total,disability_65+_1disability,disability_65+_2ormoredisabilities,disability_65+_nodisability,StateFIPS,GeoFIPS
33,1600000US4776860,0,0,None,None,None,None,None,418,93,1,0,92,300,13,6,281,25,9,4,12,47,76860
34,1600000US4778320,0,0,None,None,None,None,None,1698,439,24,3,412,992,154,82,756,267,62,73,132,47,78320
35,1600000US4778560,0,0,2245,2228,17,0,17,3976,675,86,0,589,2257,174,274,1809,1044,270,234,540,47,78560
36,1600000US4779980,0,0,None,None,None,None,None,3583,705,40,0,665,2446,276,435,1735,432,55,112,265,47,79980
37,0400000US47,1914,4297,3154232,2941235,212997,88651,124346,6664134,1505251,57989,18524,1428738,4079574,285685,252647,3541242,1079309,186133,230689,662487,47,0


In [95]:
dfs = [two,three,four,five,six,seven,eight,nine,ten,eleven,twelve,thirteen,fourteen,fifteen,sixteen,seventeen,eighteen,nineteen,twenty,twentyone,twentytwo,twentythree,
      twentyfour,twentyfive,twentysix]

In [96]:
for df in dfs:
    df = df.drop(columns = ['NAME','StateFIPS','GeoFIPS'], inplace = True)

In [97]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

The default inplace=False is intended for assigning back to the original dataframe, because it returns a new copy. However inplace=True operates on the same copy and returns None, therefore there is no need to assign back to the original dataframe.

In [98]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [103]:
#df_merged.tail()

In [101]:
dfs = [one, df_merged, last]
data = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [105]:
data.head()

,NAME,GEO_ID,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55t

In [106]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500000US47021,0500000US47147,0500000US47165,0500000US47037,0500000US47189,0500000US47169,0500000US47187,0500000US47149,0500000US47119,1600000US4702180,1600000US4709880,1600000US4713080,1600000US4715160,1600000US4720620,1600000US4722360,1600000US4728540,1600000US4732742,1600000US4739660,1600000US4741200,1600000US4741520,1600000US4744840,1600000US4750780,1600000US4751560,1600000US4752820,1600000US4757480,1600000US4759560,1600000US4769080,1600000US4769420,1600000US4776860,1600000US4778320,1600000US4778560,1600000US4779980,0400000US47
agebysex_total_series,13553,204992,8201,18528,53289,40539,70982,187680,690540,140604,10910,232380,324139,94615,4720,1711,2115,156092,15500,943,40262,10910,2744,35556,34759,1802,35834,141704,1930,2492,4660,123,50820,418,1698,4124,3583,6772268
age_total_male,6572,102205,4119,9113,26246,20221,35176,91606,332527,68930,6668,113906,159299,45700,2298,839,1130,78420,7505,472,20160,6668,1377,17716,16515,844,17691,68891,1045,1276,2281,62,25143,241,732,1988,1770,3304462
age_m_u5,353,8916,216,538,1733,1192,2325,5776,23406,4318,373,6932,10772,3200,85,46,19,7584,714,48,1416,373,99,1655,1205,96,1252,4640,79,20,51,0,1848,11,63,33,159,208551


In [107]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [108]:
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee
agebysex_total_series,13553,204992,8201,18528,53289,40539,70982,187680,690540,140604,10910,232380,324139,94615,4720,1711,2115,156092,15500,943,40262,10910,2744,35556,34759,1802,35834,141704,1930,2492,4660,123,50820,418,1698,4124,3583,6772268
age_total_male,6572,102205,4119,9113,26246,20221,35176,91606,332527,68930,6668,113906,159299,45700,2298,839,1130,78420,7505,472,20160,6668,1377,17716,16515,844,17691,68891,1045,1276,2281,62,25143,241,732,1988,1770,3304462
age_m_u5,353,8916,216,538,1733,1192,2325,5776,23406,4318,373,6932,10772,3200,85,46,19,7584,714,48,1416,373,99,1655,1205,96,1252,4640,79,20,51,0,1848,11,63,33,159,208551
age_m_5to9,476,7959,347,565,1662,1165,2352,6523,18746,4573,425,9230,11045,2504,130,59,103,6061,527,60,1537,425,91,1064,1235,23,1373,4664,41,59,182,7,2435,8,35,124,98,208045
age_m_10to14,364,7229,234,663,1850,1407,2491,6312,20115,5100,268,9690,12034,3692,198,37,61,5017,574,40,1607,268,93,1661,1129,48,1699,4978,87,86,204,7,1688,28,67,126,86,222622


In [109]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [110]:
data = transp

In [123]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [124]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [132]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']
data['Trousdale Incorporated'] = data['Hartsville/Trousdale County, Tennessee']
data['Trousdale Unincorporated'] = data['Trousdale County, Tennessee'] - data['Trousdale Incorporated']

Not dropping any places right now.

In [133]:
data.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Ashland City town, Tennessee","Burns town, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Dickson city, Tennessee","Eagleville city, Tennessee","Gallatin city, Tennessee","Hartsville/Trousdale County, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","New Johnsonville city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","White Bluff town, Tennessee",Tennessee,GNRC,MPO,Rutherford Incorporated,Rutherford Unincorporated,Wilson Incorporated,Wilson Unincorporated,Cheatham Incorporated,Cheatham Unincorporated,Dickson Incorporated,Dickson Unincorporated,Humphreys Incorporated,Humphreys Unincorporated,Montgomery Incorporated,Montgomery Unincorporated,Trousdale Incorporated,Trousdale Unincorporated
agebysex_total_series,13553.0,204992.0,8201.0,18528.0,53289.0,40539.0,70982.0,187680.0,690540.0,140604.0,10910.0,232380.0,324139.0,94615.0,4720.0,1711.0,2115.0,156092.0,15500.0,943.0,40262.0,10910.0,2744.0,35556.0,34759.0,1802.0,35834.0,141704.0,1930.0,2492.0,4660.0,123.0,50820.0,418.0,1698.0,4124.0,3583.0,6772268.0,1996337.0,1740940.0,229023.0,95116.0,72291.0,68313.0,14616.0,25923.0,23450.0,29839.0,7856.0,10672.0,156092.0,48900.0,10910.0,0.0
age_total_male,6572.0,102205.0,4119.0,9113.0,26246.0,20221.0,35176.0,91606.0,332527.0,68930.0,6668.0,113906.0,159299.0,45700.0,2298.0,839.0,1130.0,78420.0,7505.0,472.0,20160.0,6668.0,1377.0,17716.0,16515.0,844.0,17691.0,68891.0,1045.0,1276.0,2281.0,62.0,25143.0,241.0,732.0,1988.0,1770.0,3304462.0,976588.0,847144.0,112222.0,47077.0,34938.0,33992.0,7232.0,12989.0,11547.0,14699.0,3877.0,5236.0,78420.0,23785.0,6668.0,0.0
age_m_u5,353.0,8916.0,216.0,538.0,1733.0,1192.0,2325.0,5776.0,23406.0,4318.0,373.0,6932.0,10772.0,3200.0,85.0,46.0,19.0,7584.0,714.0,48.0,1416.0,373.0,99.0,1655.0,1205.0,96.0,1252.0,4640.0,79.0,20.0,51.0,0.0,1848.0,11.0,63.0,33.0,159.0,208551.0,66850.0,56729.0,8191.0,2581.0,2520.0,1798.0,255.0,937.0,949.0,784.0,208.0,330.0,7584.0,1332.0,373.0,0.0
age_m_5to9,476.0,7959.0,347.0,565.0,1662.0,1165.0,2352.0,6523.0,18746.0,4573.0,425.0,9230.0,11045.0,2504.0,130.0,59.0,103.0,6061.0,527.0,60.0,1537.0,425.0,91.0,1064.0,1235.0,23.0,1373.0,4664.0,41.0,59.0,182.0,7.0,2435.0,8.0,35.0,124.0,98.0,208045.0,65068.0,54973.0,8223.0,2822.0,2643.0,1930.0,462.0,703.0,802.0,860.0,188.0,377.0,6061.0,1898.0,425.0,0.0
age_m_10to14,364.0,7229.0,234.0,663.0,1850.0,1407.0,2491.0,6312.0,20115.0,5100.0,268.0,9690.0,12034.0,3692.0,198.0,37.0,61.0,5017.0,574.0,40.0,1607.0,268.0,93.0,1661.0,1129.0,48.0,1699.0,4978.0,87.0,86.0,204.0,7.0,1688.0,28.0,67.0,126.0,86.0,222622.0,67757.0,59434.0,8367.0,3667.0,2895.0,2205.0,581.0,826.0,793.0,1057.0,261.0,402.0,5017.0,2212.0,268.0,0.0


In [134]:
ok = data.transpose().reset_index()

In [135]:
ok.head()

,NAME,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_wpr

In [136]:
ok.tail()

,NAME,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_wpr

In [137]:
ok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Columns: 1219 entries, NAME to GeoFIPS
dtypes: float64(1218), object(1)
memory usage: 514.4+ KB


In [138]:
ok.to_csv('../data/ACS20205YR.csv', index = False)